Install Delta Lake:-

In [ ]:
!pip install -q pyspark==3.5.1 delta-spark==3.2.0

Create Spark Session:-

In [ ]:
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession

builder = SparkSession.builder \
    .appName("DeltaAssignment") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()

Delta Lake Incremental Data Processing Assignment

Objective:-

Load dataset into Delta Table

Clean data

Create Incremental Dataset

MERGE (Update + Insert)

Validation

Final Output

Import Libraries:-

In [ ]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable

Upload CSV:-

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving Sample - Superstore.csv to Sample - Superstore (1).csv


Load CSV

In [ ]:
df = spark.read.csv(
    "Sample - Superstore.csv",
    header=True,
    inferSchema=True
)

df.show(10, truncate=False)

print("Rows :", df.count())
print("Columns :", len(df.columns))

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+----------------------------------------------------------------+--------+--------+--------+--------+
|Row ID|Order ID      |Order Date|Ship Date |Ship Mode     |Customer ID|Customer Name  |Segment  |Country      |City           |State     |Postal Code|Region|Product ID     |Category       |Sub-Category|Product Name                                                    |Sales   |Quantity|Discount|Profit  |
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+----------------------------------------------------------------+--------+--------+--------+--------+
|1     |CA-2016-152156|11/8/2016 |11/11/2016|Second Class  |CG-12520   |Claire Gute  

Data Cleaning

Removing null values and duplicate records.

Remove Null Values:-

In [ ]:
clean_df = df.dropna()

Remove Duplicate Records:-

In [ ]:
clean_df = clean_df.dropDuplicates()

print("Rows after cleaning:", clean_df.count())

clean_df.show(10, truncate=False)

Rows after cleaning: 9994
+------+--------------+----------+----------+--------------+-----------+----------------+-----------+-------------+-------------+------------+-----------+-------+---------------+---------------+------------+---------------------------------------------------------------------------------+-------+--------+--------+-------+
|Row ID|Order ID      |Order Date|Ship Date |Ship Mode     |Customer ID|Customer Name   |Segment    |Country      |City         |State       |Postal Code|Region |Product ID     |Category       |Sub-Category|Product Name                                                                     |Sales  |Quantity|Discount|Profit |
+------+--------------+----------+----------+--------------+-----------+----------------+-----------+-------------+-------------+------------+-----------+-------+---------------+---------------+------------+---------------------------------------------------------------------------------+-------+--------+--------+-------+
|1

In [ ]:
# Replace spaces with underscores in column names
clean_df = clean_df.toDF(*[c.replace(" ", "_") for c in clean_df.columns])

# Verify new column names
print(clean_df.columns)

['Row_ID', 'Order_ID', 'Order_Date', 'Ship_Date', 'Ship_Mode', 'Customer_ID', 'Customer_Name', 'Segment', 'Country', 'City', 'State', 'Postal_Code', 'Region', 'Product_ID', 'Category', 'Sub-Category', 'Product_Name', 'Sales', 'Quantity', 'Discount', 'Profit']


Create Delta Table

Store cleaned dataset in Delta Format.

Save Delta Table

In [ ]:
delta_path = "/content/delta/superstore"

clean_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(delta_path)

Load Delta Table

In [ ]:
from delta.tables import DeltaTable

deltaTable = DeltaTable.forPath(spark, delta_path)

deltaTable.toDF().show(10, truncate=False)

+------+--------------+----------+----------+--------------+-----------+----------------+-----------+-------------+-------------+------------+-----------+-------+---------------+---------------+------------+---------------------------------------------------------------------------------+-------+--------+--------+-------+
|Row_ID|Order_ID      |Order_Date|Ship_Date |Ship_Mode     |Customer_ID|Customer_Name   |Segment    |Country      |City         |State       |Postal_Code|Region |Product_ID     |Category       |Sub-Category|Product_Name                                                                     |Sales  |Quantity|Discount|Profit |
+------+--------------+----------+----------+--------------+-----------+----------------+-----------+-------------+-------------+------------+-----------+-------+---------------+---------------+------------+---------------------------------------------------------------------------------+-------+--------+--------+-------+
|188   |CA-2016-157000|7/16/

Create Incremental Dataset

Simulate updated and new records.

Incremental Dataset

In [ ]:
incremental_data = [

(2,
"CA-2016-152156",
"11/8/2016",
"11/11/2016",
"Second Class",
"CG-12520",
"Claire Gute",
"Consumer",
"United States",
"Henderson",
"Kentucky",
42420,
"South",
"FUR-CH-10000454",
"Furniture",
"Chairs",
"Updated Chair",
350.00,
3,
0,
120),

(10001,
"CA-2018-999999",
"12/15/2018",
"12/18/2018",
"Standard Class",
"AB-12345",
"John Smith",
"Corporate",
"United States",
"New York",
"New York",
10001,
"East",
"OFF-PA-999999",
"Office Supplies",
"Paper",
"Premium Paper",
125.50,
5,
0,
55)

]

incremental_df = spark.createDataFrame(
    incremental_data,
    clean_df.columns
)

incremental_df.show(truncate=False)

+------+--------------+----------+----------+--------------+-----------+-------------+---------+-------------+---------+--------+-----------+------+---------------+---------------+------------+-------------+-----+--------+--------+------+
|Row_ID|Order_ID      |Order_Date|Ship_Date |Ship_Mode     |Customer_ID|Customer_Name|Segment  |Country      |City     |State   |Postal_Code|Region|Product_ID     |Category       |Sub-Category|Product_Name |Sales|Quantity|Discount|Profit|
+------+--------------+----------+----------+--------------+-----------+-------------+---------+-------------+---------+--------+-----------+------+---------------+---------------+------------+-------------+-----+--------+--------+------+
|2     |CA-2016-152156|11/8/2016 |11/11/2016|Second Class  |CG-12520   |Claire Gute  |Consumer |United States|Henderson|Kentucky|42420      |South |FUR-CH-10000454|Furniture      |Chairs      |Updated Chair|350.0|3       |0       |120   |
|10001 |CA-2018-999999|12/15/2018|12/18/2018

MERGE Operation

Update existing records and insert new records.

MERGE:-

In [ ]:
deltaTable.alias("target") \
.merge(
incremental_df.alias("source"),
"target.Row_ID = source.Row_ID"
) \
.whenMatchedUpdateAll() \
.whenNotMatchedInsertAll() \
.execute()

In [ ]:
print("MERGE Completed Successfully")

MERGE Completed Successfully


Final Delta Table

In [ ]:
final_df = deltaTable.toDF()

final_df.show(20, truncate=False)

+------+--------------+----------+----------+--------------+-----------+----------------+-----------+-------------+-------------+--------------+-----------+-------+---------------+---------------+------------+--------------------------------------------------------------------+-------+--------+--------+--------+
|Row_ID|Order_ID      |Order_Date|Ship_Date |Ship_Mode     |Customer_ID|Customer_Name   |Segment    |Country      |City         |State         |Postal_Code|Region |Product_ID     |Category       |Sub-Category|Product_Name                                                        |Sales  |Quantity|Discount|Profit  |
+------+--------------+----------+----------+--------------+-----------+----------------+-----------+-------------+-------------+--------------+-----------+-------+---------------+---------------+------------+--------------------------------------------------------------------+-------+--------+--------+--------+
|26    |CA-2016-121755|1/16/2016 |1/20/2016 |Second Class 

Validation

Validate row count, duplicates and null values.

Row Count

In [ ]:
print("Total Rows :", final_df.count())

Total Rows : 9995


Duplicate Check

In [ ]:
duplicates = final_df.groupBy("Row_ID") \
.count() \
.filter(col("count")>1)

duplicates.show()

+------+-----+
|Row_ID|count|
+------+-----+
+------+-----+



Null Check

In [ ]:
final_df.select([
sum(when(col(c).isNull(),1).otherwise(0)).alias(c)
for c in final_df.columns
]).show()

+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|Row_ID|Order_ID|Order_Date|Ship_Date|Ship_Mode|Customer_ID|Customer_Name|Segment|Country|City|State|Postal_Code|Region|Product_ID|Category|Sub-Category|Product_Name|Sales|Quantity|Discount|Profit|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|     0|       0|         0|        0|        0|          0|            0|      0|      0|   0|    0|          0|     0|         0|       0|           0|           0|    0|       0|       0|     0|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+



Incremental processing completed successfully using Delta Lake.

Final Summary

In [ ]:
print("="*50)

print("Assignment Completed Successfully")

print("="*50)

print("Final Row Count :", final_df.count())

print("Duplicate Records :", duplicates.count())

print("MERGE Completed Successfully")

print("="*50)

Assignment Completed Successfully
Final Row Count : 9995
Duplicate Records : 0
MERGE Completed Successfully


# Export Datasets for GitHub Submission

In [ ]:
clean_df.coalesce(1) \
.write \
.mode("overwrite") \
.option("header",True) \
.csv("/content/superstore_master")

In [ ]:
incremental_df.coalesce(1) \
.write \
.mode("overwrite") \
.option("header",True) \
.csv("/content/superstore_incremental")

In [ ]:
import glob
import shutil

master = glob.glob("/content/superstore_master/part-*.csv")[0]
shutil.copy(master,"/content/superstore_master.csv")

incremental = glob.glob("/content/superstore_incremental/part-*.csv")[0]
shutil.copy(incremental,"/content/superstore_incremental.csv")

print("CSV Files Created Successfully")

CSV Files Created Successfully


In [ ]:
from google.colab import files

files.download("/content/superstore_master.csv")
files.download("/content/superstore_incremental.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>